# SD3.5 CityPersons Augmentation Runner

Runner notebook gon cho Kaggle. Notebook nay clone repo tu GitHub vao `/kaggle/working/VIN`, sau do import truc tiep cac module `sd35_*.py` tu repo da clone.

## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" sentencepiece protobuf safetensors ultralytics opencv-python


## 2. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/BDT-17/VIN.git'
REPO_DIR = Path('/kaggle/working/VIN')

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR / 'notebooks'
if not (PROJECT_DIR / 'sd35_config.py').exists():
    PROJECT_DIR = REPO_DIR

%cd {PROJECT_DIR}
print('PROJECT_DIR:', PROJECT_DIR)


## 3. Imports

In [ ]:

import sys
import importlib
from pathlib import Path

module_dirs = [str(PROJECT_DIR), str(Path('/kaggle/working'))]
sys.path = module_dirs + [path for path in sys.path if path not in module_dirs]

for module_name in list(sys.modules):
    if module_name.startswith('sd35_'):
        del sys.modules[module_name]

from sd35_config import *
from sd35_metrics import *
from sd35_data import *
from sd35_utils import *
from sd35_model import *
from sd35_evaluation import *
from sd35_pipeline import *
from sd35_runner import *
import sd35_runner

ensure_output_dirs()
print('sd35_runner:', sd35_runner.__file__)
print('generation pipeline:', CONTEXT_PERSON_GENERATION_PIPELINE)
print('output dir:', OUTPUT_DIR)


## 4. Runtime Check

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu count:', torch.cuda.device_count())
    for index in range(torch.cuda.device_count()):
        print(index, torch.cuda.get_device_name(index))


## 5. Hugging Face Login

In [ ]:
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    import os
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError("HF_TOKEN not found. Add a Kaggle secret named HF_TOKEN or set os.environ['HF_TOKEN'].") from exc

login(token=hf_token)
print("Hugging Face login OK from HF_TOKEN secret.")


## 6. Dataset Scan

In [ ]:
records = load_records()
summarize_citypersons_records(records)
preview_prompt_samples(records)


## 8. Smoke Test Across Added Datasets

Run these cells to smoke-test the three configured dataset layouts. Each run switches the runtime dataset paths and writes outputs into a separate folder under `/kaggle/working/sd35_smoke/`.

In [ ]:
from pathlib import Path
import sd35_config as cfg
import sd35_data as data_mod
import sd35_utils as utils_mod
import sd35_pipeline as pipeline_mod
import sd35_evaluation as eval_mod
import sd35_edge_harmonization as edge_mod
import sd35_runner as runner_mod
import sd35_metrics as metrics_mod

DATASET_SMOKE_RUNS = {
    "citypersons_bg_yolo": {
        "root": Path("/kaggle/input/datasets/muttahirulislam/citypersons-dataset-with-bg-image/yolo_dir/yolo_dir"),
        "splits": ["train"],
    },
    "cityperson_nguyena": {
        "root": Path("/kaggle/input/datasets/nguyenaabcxyzeric/cityperson"),
        "splits": ["test"],
    },
    "mot17_02_frcnn": {
        "root": Path("/kaggle/input/datasets/kyoru4444/mot17-02-fcrnn/MOT17-02-FRCNN"),
        "splits": ["test"],
        "overrides": {
            # MOT17 is crowded and has many partial/overlapping pedestrians.
            # Keep detector validation, but make placement/overlap less brittle for smoke runs.
            "USE_SEMANTIC_PLACEMENT": False,
            "REQUIRE_SEMANTIC_PLACEMENT": False,
            "ALLOW_PERSON_PERSON_OVERLAP": True,
            "MAX_PERSON_PERSON_OVERLAP_RATIO": 0.18,
            "PERSON_OVERLAP_MIN_FRONT_HEIGHT_RATIO": 1.06,
            "OCCLUDED_PERSON_MAX_HEIGHT_RATIO": 0.88,
            "OCCLUDED_PERSON_MAX_FOOT_Y_DELTA": 12,
            "PATCH_ROAD_Y_RANGE": (0.58, 0.94),
            "MIN_FOOT_SUPPORT": 0.28,
            "MAX_BODY_AVOID_SUPPORT": 0.28,
            "CONTEXT_GENERATION_RETRIES": 5,
            "MIN_RETRY_PERSON_CONFIDENCE": 0.20,
            "CONTEXT_PERSON_MIN_CONFIDENCE": 0.10,
        },
    },
    "human_detection_dataset": {
        "root": Path("/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset"),
        "image_dir": Path("/kaggle/input/datasets/constantinwerner/human-detection-dataset/human detection dataset/0"),
        "splits": ["train"],
    },
}

def _mot_root(path):
    path = Path(path)
    return path.parent if path.name == "img1" else path

def _is_mot_root(path):
    path = _mot_root(path)
    return (path / "img1").exists() and ((path / "gt").exists() or (path / "det").exists())

def _split_dirs_for_root(root):
    root = _mot_root(root)
    if _is_mot_root(root):
        return {"test": root / "img1"}, {"test": root / "gt"}
    valid_split = "valid" if (root / "valid" / "images").exists() else "val"
    return (
        {
            "train": root / "train" / "images",
            "val": root / valid_split / "images",
            "test": root / "test" / "images",
        },
        {
            "train": root / "train" / "labels",
            "val": root / valid_split / "labels",
            "test": root / "test" / "labels",
        },
    )

def configure_dataset_smoke(dataset_name):
    spec = DATASET_SMOKE_RUNS[dataset_name]
    root = _mot_root(spec["root"])
    if "image_dir" in spec:
        image_dir = Path(spec["image_dir"])
        split_name = spec["splits"][0]
        split_dirs = {split_name: image_dir}
        label_dirs = {split_name: image_dir / "labels"}
    else:
        split_dirs, label_dirs = _split_dirs_for_root(root)
    active_split_dirs = {split: split_dirs[split] for split in spec["splits"]}
    missing = [str(path) for path in active_split_dirs.values() if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing dataset paths for {dataset_name}: {missing}")

    output_dir = Path("/kaggle/working/sd35_smoke") / dataset_name
    patch_debug_dir = output_dir / "patch_debug"
    edge_debug_dir = output_dir / "edge_harmonization_debug"
    for path in (output_dir, patch_debug_dir, edge_debug_dir):
        path.mkdir(parents=True, exist_ok=True)

    runtime_modules = (cfg, data_mod, utils_mod, pipeline_mod, eval_mod, edge_mod, runner_mod, metrics_mod)
    for mod in runtime_modules:
        mod.DATASET_ROOT = root
        mod.IMAGE_ROOT = root
        mod.LABEL_ROOT = root
        mod.DATASET_SPLIT_DIRS = split_dirs
        mod.LABEL_SPLIT_DIRS = label_dirs
        mod.TARGET_SPLITS = list(spec["splits"])
        mod.OUTPUT_DIR = output_dir
        mod.PATCH_DEBUG_DIR = patch_debug_dir
        mod.EDGE_DEBUG_DIR = edge_debug_dir
    for name, value in spec.get("overrides", {}).items():
        for mod in runtime_modules:
            setattr(mod, name, value)
        cfg.EFFECTIVE_CONFIG[name] = value

    print(f"dataset={dataset_name}")
    print(f"root={root}")
    print(f"splits={spec['splits']}")
    print(f"output_dir={output_dir}")
    records = data_mod.scan_dataset(split_dirs=active_split_dirs)
    data_mod.summarize_citypersons_records(records)
    return records, spec

def run_dataset_smoke(dataset_name, smoke_images=3):
    records, spec = configure_dataset_smoke(dataset_name)
    generated_paths, manifest_rows, autotune_report = runner_mod.run_smoke(
        records,
        smoke_images=smoke_images,
        smoke_splits=spec["splits"],
    )
    return generated_paths, manifest_rows, autotune_report


def show_dataset_metrics(dataset_name):
    metrics_path = Path("/kaggle/working/sd35_smoke") / dataset_name / "metrics_summary.json"
    csv_path = metrics_path.with_suffix(".csv")
    if not metrics_path.exists():
        raise FileNotFoundError(f"Metrics summary not found yet. Run the {dataset_name} smoke cell first: {metrics_path}")
    summary = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("metrics_summary:", summary)
    print("metrics_summary_json:", metrics_path)
    print("metrics_summary_csv:", csv_path)
    return summary


### 8.1 CityPersons YOLO Smoke Test

In [ ]:
citypersons_bg_yolo_paths, citypersons_bg_yolo_rows, citypersons_bg_yolo_autotune = run_dataset_smoke(
    "citypersons_bg_yolo",
    smoke_images=3,
)
citypersons_bg_yolo_paths[:5]


#### CityPersons YOLO Metrics


In [ ]:
citypersons_bg_yolo_metrics = show_dataset_metrics("citypersons_bg_yolo")


### 8.2 CityPerson NguyenAabcxyzEric Smoke Test

In [ ]:
cityperson_nguyena_paths, cityperson_nguyena_rows, cityperson_nguyena_autotune = run_dataset_smoke(
    "cityperson_nguyena",
    smoke_images=3,
)
cityperson_nguyena_paths[:5]


#### CityPerson Nguyen Metrics


In [ ]:
cityperson_nguyena_metrics = show_dataset_metrics("cityperson_nguyena")


### 8.3 MOT17-02-FRCNN Smoke Test

In [ ]:
mot17_02_frcnn_paths, mot17_02_frcnn_rows, mot17_02_frcnn_autotune = run_dataset_smoke(
    "mot17_02_frcnn",
    smoke_images=3,
)
mot17_02_frcnn_paths[:5]


#### MOT17-02-FRCNN Metrics


In [ ]:
mot17_02_frcnn_metrics = show_dataset_metrics("mot17_02_frcnn")


### 8.4 Human Detection Dataset Smoke Test


In [ ]:
human_detection_paths, human_detection_rows, human_detection_autotune = run_dataset_smoke(
    "human_detection_dataset",
    smoke_images=3,
)
human_detection_paths[:5]


#### Human Detection Dataset Metrics


In [ ]:
human_detection_dataset_metrics = show_dataset_metrics("human_detection_dataset")


## 9. Export Outputs

In [ ]:
from sd35_lora_training import run_lora_training

# Dry-run by default: writes training_config.json and train_command.json only.
# Set LORA_TRAINING_ENABLED=True in sd35_config.py when you want to launch training.
lora_training_artifacts = run_lora_training(dry_run=not bool(LORA_TRAINING_ENABLED))
lora_training_artifacts


In [ ]:
# Run after generation if you want a zip artifact.
export_outputs()
